[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/09-system-design/08_ids.ipynb)

*System Design · Notebook 8 of 10*

# URL Shortener & ID Generation

The classic first design question: base62 codes, code length and capacity, hashing with collision handling, ticket ranges and key-generation services, Snowflake IDs (compose, decode, sequence limits, clock rollback), time-sortable IDs, birthday-paradox collisions, Bloom filters, aliases, expiry, analytics and URL canonicalisation.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A Snowflake ID is a timestamped ticket that also says which counter printed it, so no two counters ever hand out the same number.

**What interviewers really test here:** Base62, code capacity, hashing vs counters vs key services, Snowflake layout, clock issues, Bloom filters and 301 vs 302.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Design a URL shortener *(Basics)*
- [ ] 2. Generating short codes *(Core)*
- [ ] 3. Unique IDs at scale *(Core)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### ID schemes compared

| Scheme | Size | Sortable | Coordination |
| --- | --- | --- | --- |
| Auto-increment | 64 bit | yes | one database |
| UUIDv4 | 128 bit | no | none |
| Snowflake | 64 bit | by time | unique machine ids |
| ULID / UUIDv7 | 128 bit | by time | none |

---
# Part 2: Lessons, step by step

## Lesson 1: Design a URL shortener

*Level: Basics*

**Requirements**: shorten a long URL, redirect fast, optional custom alias and expiry, click analytics. Reads outnumber writes ~100:1.



```python
POST /v1/links {long_url, alias?, ttl?} → 201 {code}
GET  /{code}  → 301/302 Location: long_url
```



**High level**: stateless API servers → cache (Redis, hot codes) → key-value store (code → long URL), sharded by code. Analytics go through a queue so redirects stay fast. A 7-character base62 code gives 3.5 trillion links.



**Deep dives**: how to generate codes (below), 301 vs 302 (302 keeps every click visible for analytics), expiry, abuse (rate limits, malware checks).

**✏️ Your turn:** you now have what you need for [Q9](#q9), [Q3](#q3), [Q6](#q6), [Q7](#q7), [Q8](#q8), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 2: Generating short codes

*Level: Core*

| Approach | Pros | Cons |
| --- | --- | --- |
| Counter + base62 | no collisions, short | needs a shared counter; guessable |
| Counter ranges per server | no per-request coordination | ids out of global order; lost ranges on crash |
| Hash of URL (first k chars) | same URL → same code | collisions to detect and retry |
| Key-generation service | pre-made random keys, fast | another service; crash wastes batches |



A **Bloom filter** answers "is this code taken?" in memory: "no" is certain, "maybe" needs a database check.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q12](#q12), [Q13](#q13), [Q14](#q14), [Q19](#q19), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 3: Unique IDs at scale

*Level: Core*

Auto-increment ids need one database. Alternatives:




- **UUIDv4**: 122 random bits, no coordination, but random order hurts B-tree index locality.
- **Snowflake**: 41-bit ms timestamp + 10-bit machine + 12-bit sequence in 64 bits — roughly time-ordered, 4096 ids per ms per machine. Machine ids must be unique, and the clock must never go backwards.
- **ULID / UUIDv7**: timestamp prefix + randomness — sortable strings that index well.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def snowflake_demo():
    epoch = 1_700_000_000_000
    ms, machine, seq = 1_700_000_123_456, 7, 3
    sid = (ms - epoch) << 22 | machine << 12 | seq
    print("id:", sid)
    print("timestamp:", (sid >> 22) + epoch, "machine:", (sid >> 12) & 1023, "sequence:", sid & 4095)


snowflake_demo()

id: 517812023299
timestamp: 1700000123456 machine: 7 sequence: 3


**✏️ Your turn:** you now have what you need for [Q15](#q15), [Q4](#q4), [Q5](#q5), [Q16](#q16), [Q10](#q10), [Q17](#q17), [Q18](#q18). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [3]:
ALPHABET = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"

EPOCH = 1_700_000_000_000

from collections import Counter, defaultdict


<a id="q1"></a>
### Q1. Base62 Encode

🟢 Warm-up · *Numbers to short codes*

Encode each non-negative integer in base 62 using `ALPHABET` (0 → '0').

**Examples**

- `nums = [0, 61, 62, 125, 3843, 1000000, 56800235583]` → `['0', 'Z', '10', '21', 'ZZ', '4c92', 'ZZZZZZ']`

In [ ]:
def b62_encode(nums):
    """
    Encode each non-negative integer in base 62 using ALPHABET (0 → '0').
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(b62_encode, [
    (([0, 61, 62, 125, 3843, 1000000, 56800235583],), ['0', 'Z', '10', '21', 'ZZ', '4c92', 'ZZZZZZ']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeated division, building a list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`divmod` in a helper.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Repeated division, building a list.

```python
def b62_encode(nums):
    out = []
    for n in nums:
        if n == 0:
            out.append("0")
            continue
        s = ""
        while n > 0:
            s = ALPHABET[n % 62] + s
            n = n // 62
        out.append(s)
    return out
```

**Cost:** O(n · log v) time, O(log v) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `divmod` in a helper.

```python
def b62_encode(nums):
    def enc(n):
        s = []
        while True:
            n, r = divmod(n, 62)
            s.append(ALPHABET[r])
            if n == 0:
                return "".join(reversed(s))
    return [enc(n) for n in nums]
```

**Cost:** O(n · log v) time, O(log v) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeated division, building a list. That costs O(n · log v) time and O(log v) space."

"I can do better: `divmod` in a helper. That's O(n · log v) time and O(log v) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/base62-encode)

<a id="q2"></a>
### Q2. Base62 Decode

🟢 Warm-up · *Short codes back to numbers*

Decode each base62 code back to its integer.

**Examples**

- `codes = ['0', 'Z', '10', '21', 'ZZ', '4c92', 'ZZZZZZ']` → `[0, 61, 62, 125, 3843, 1000000, 56800235583]`

In [ ]:
def b62_decode(codes):
    """
    Decode each base62 code back to its integer.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(b62_decode, [
    ((['0', 'Z', '10', '21', 'ZZ', '4c92', 'ZZZZZZ'],), [0, 61, 62, 125, 3843, 1000000, 56800235583]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Search the alphabet for each character.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A char → value dict and Horner's rule.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Search the alphabet for each character.

```python
def b62_decode(codes):
    out = []
    for c in codes:
        n = 0
        for ch in c:
            n = n * 62 + ALPHABET.index(ch)
        out.append(n)
    return out
```

**Cost:** O(n · L · 62) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A char → value dict and Horner's rule.

```python
def b62_decode(codes):
    value = {ch: i for i, ch in enumerate(ALPHABET)}
    out = []
    for c in codes:
        n = 0
        for ch in c:
            n = n * 62 + value[ch]
        out.append(n)
    return out
```

**Cost:** O(n · L) time, O(62) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Search the alphabet for each character. That costs O(n · L · 62) time and O(1) space."

"I can do better: A char → value dict and Horner's rule. That's O(n · L) time and O(62) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/base62-decode)

<a id="q3"></a>
### Q3. How Long Will the Codes Last?

🟢 Warm-up · *Capacity planning*

For each code length, return `(number of codes, years until exhausted)` at `per_day` new links (years rounded to 1).

**Examples**

- `lengths = [5, 6, 7, 8], per_day = 100000000` → `[(916132832, 0.0), (56800235584, 1.6), (3521614606208, 96.5), (218340105584896, 5981.9)]`

In [ ]:
def code_capacity(lengths, per_day):
    """
    For each code length, return (number of codes, years until exhausted) at
    per_day new links (years rounded to 1).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(code_capacity, [
    (([5, 6, 7, 8], 100000000), [(916132832, 0.0), (56800235584, 1.6), (3521614606208, 96.5), (218340105584896, 5981.9)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply 62 in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`62 ** L`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Multiply 62 in a loop.

```python
def code_capacity(lengths, per_day):
    out = []
    for L in lengths:
        n = 1
        for _ in range(L):
            n *= 62
        out.append((n, round(n / per_day / 365, 1)))
    return out
```

**Cost:** O(L) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `62 ** L`.

```python
def code_capacity(lengths, per_day):
    return [(62 ** L, round(62 ** L / per_day / 365, 1)) for L in lengths]
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply 62 in a loop. That costs O(L) time and O(1) space."

"I can do better: `62 ** L`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/how-long-will-the-codes-last)

<a id="q4"></a>
### Q4. Decode a Snowflake ID

🟢 Warm-up · *Read the timestamp back*

Return `(unix_ms, machine, sequence)` for each id.

**Examples**

- `ids = [0, 4198401, 4202497, 281964602986594303]` → `[(1700000000000, 0, 0), (1700000000001, 1, 1), (1700000000001, 2, 1), (1767225600000, 1023, 4095)]`

In [ ]:
def unsnowflake(ids):
    """
    Return (unix_ms, machine, sequence) for each id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(unsnowflake, [
    (([0, 4198401, 4202497, 281964602986594303],), [(1700000000000, 0, 0), (1700000000001, 1, 1), (1700000000001, 2, 1), (1767225600000, 1023, 4095)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Integer division and modulo.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Shifts and masks.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Integer division and modulo.

```python
def unsnowflake(ids):
    out = []
    for i in ids:
        out.append((i // 2 ** 22 + EPOCH, i // 2 ** 12 % 1024, i % 4096))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Shifts and masks.

```python
def unsnowflake(ids):
    return [((i >> 22) + EPOCH, (i >> 12) & 0x3FF, i & 0xFFF) for i in ids]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Integer division and modulo. That costs O(n) time and O(n) space."

"I can do better: Shifts and masks. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/decode-a-snowflake-id)

<a id="q5"></a>
### Q5. When the Sequence Runs Out

🟢 Warm-up · *IDs per millisecond*

With `seq_bits` sequence bits, one machine makes 2^seq\_bits ids per ms; extra requests wait for the next ms. For each burst size (requests arriving in the same ms), return how many milliseconds it takes to issue them.

**Examples**

- `bursts = [1, 4096, 4097, 10000, 50000], seq_bits = 12` → `[1, 1, 2, 3, 13]`

In [ ]:
def seq_overflow(bursts, seq_bits):
    """
    With seq_bits sequence bits, one machine makes 2^seq_bits ids per ms; extra
    requests wait for the next ms. For each burst size (requests arriving in the
    same ms), return how many milliseconds it takes to issue them.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(seq_overflow, [
    (([1, 4096, 4097, 10000, 50000], 12), [1, 1, 2, 3, 13]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Subtract per-ms capacity in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Ceiling division.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Subtract per-ms capacity in a loop.

```python
def seq_overflow(bursts, seq_bits):
    out = []
    for b in bursts:
        ms = 0
        while b > 0:
            b -= 2 ** seq_bits
            ms += 1
        out.append(ms)
    return out
```

**Cost:** O(n · b) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Ceiling division.

```python
def seq_overflow(bursts, seq_bits):
    return [math.ceil(b / (1 << seq_bits)) for b in bursts]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Subtract per-ms capacity in a loop. That costs O(n · b) time and O(1) space."

"I can do better: Ceiling division. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/when-the-sequence-runs-out)

<a id="q6"></a>
### Q6. Validate Custom Aliases

🟢 Warm-up · *User-chosen short links*

An alias must be 4–20 characters of letters, digits or '-', not in `RESERVED` (case-insensitive) and not already in `taken`. Return 'ok', 'invalid', 'reserved' or 'taken' for each.

**Examples**

- `requests = ['my-shop', 'ab', 'Admin', 'sale_2025', 'promo', 'summer-sale'], taken = {'promo'}` → `['ok', 'invalid', 'reserved', 'invalid', 'taken', 'ok']`

In [ ]:
import re

RESERVED = {"api", "admin", "login", "help"}

def aliases(requests, taken):
    """
    An alias must be 4–20 characters of letters, digits or '-', not in RESERVED
    (case-insensitive) and not already in taken. Return 'ok', 'invalid',
    'reserved' or 'taken' for each.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(aliases, [
    ((['my-shop', 'ab', 'Admin', 'sale_2025', 'promo', 'summer-sale'], {'promo'}), ['ok', 'invalid', 'reserved', 'invalid', 'taken', 'ok']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check characters one by one.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regular expression.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Check characters one by one.

```python
def aliases(requests, taken):
    allowed = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-"
    out = []
    for a in requests:
        if not (4 <= len(a) <= 20) or any(ch not in allowed for ch in a):
            out.append("invalid")
        elif a.lower() in RESERVED:
            out.append("reserved")
        elif a in taken:
            out.append("taken")
        else:
            out.append("ok")
    return out
```

**Cost:** O(n · L) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regular expression.

```python
def aliases(requests, taken):
    pattern = re.compile(r"[A-Za-z0-9-]{4,20}")
    return ["invalid" if not pattern.fullmatch(a) else "reserved" if a.lower() in RESERVED
            else "taken" if a in taken else "ok" for a in requests]
```

**Cost:** O(n · L) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check characters one by one. That costs O(n · L) time and O(1) space."

"I can do better: A regular expression. That's O(n · L) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/validate-custom-aliases)

<a id="q7"></a>
### Q7. Expire Old Links

🟢 Warm-up · *TTL cleanup*

Links are `(code, created_day, ttl_days or None)`. On day `today`, return the expired codes (a link with ttl t expires on day created + t).

**Examples**

- `links = [('a1', 0, 30), ('b2', 10, 7), ('c3', 5, None), ('d4', 20, 30), ('e5', 1, 49)], today = 50` → `['a1', 'b2', 'd4', 'e5']`

In [ ]:
def expired(links, today):
    """
    Links are (code, created_day, ttl_days or None). On day today, return the
    expired codes (a link with ttl t expires on day created + t).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(expired, [
    (([('a1', 0, 30), ('b2', 10, 7), ('c3', 5, None), ('d4', 20, 30), ('e5', 1, 49)], 50), ['a1', 'b2', 'd4', 'e5']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop.

```python
def expired(links, today):
    out = []
    for code, created, ttl in links:
        if ttl is not None and created + ttl <= today:
            out.append(code)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def expired(links, today):
    return [c for c, d, ttl in links if ttl is not None and d + ttl <= today]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(n) time and O(n) space."

"I can do better: A comprehension. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/expire-old-links)

<a id="q8"></a>
### Q8. Click Analytics

🟢 Warm-up · *Aggregate redirect events*

Clicks are `(code, country)`. Return, per code (sorted), the total clicks and the top country (ties: alphabetical).

**Examples**

- `clicks = [('x1', 'IN'), ('x1', 'US'), ('x2', 'DE'), ('x1', 'IN'), ('x2', 'DE'), ('x2', 'FR'), ('x1', 'US'), ('x3', 'IN')]` → `[('x1', 4, 'IN'), ('x2', 3, 'DE'), ('x3', 1, 'IN')]`

In [ ]:
def click_stats(clicks):
    """
    Clicks are (code, country). Return, per code (sorted), the total clicks and
    the top country (ties: alphabetical).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(click_stats, [
    (([('x1', 'IN'), ('x1', 'US'), ('x2', 'DE'), ('x1', 'IN'), ('x2', 'DE'), ('x2', 'FR'), ('x1', 'US'), ('x3', 'IN')],), [('x1', 4, 'IN'), ('x2', 3, 'DE'), ('x3', 1, 'IN')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested dicts of counts.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`defaultdict(Counter)`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Nested dicts of counts.

```python
def click_stats(clicks):
    per = {}
    for code, country in clicks:
        per.setdefault(code, {})
        per[code][country] = per[code].get(country, 0) + 1
    out = []
    for code in sorted(per):
        counts = per[code]
        top = sorted(counts, key=lambda c: (-counts[c], c))[0]
        out.append((code, sum(counts.values()), top))
    return out
```

**Cost:** O(n + c log c) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `defaultdict(Counter)`.

```python
def click_stats(clicks):
    per = defaultdict(Counter)
    for code, country in clicks:
        per[code][country] += 1
    return [(code, sum(c.values()), min(c, key=lambda k: (-c[k], k))) for code, c in sorted(per.items())]
```

**Cost:** O(n + c log c) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested dicts of counts. That costs O(n + c log c) time and O(n) space."

"I can do better: `defaultdict(Counter)`. That's O(n + c log c) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/click-analytics)

<a id="q9"></a>
### Q9. Napkin Maths for a Shortener

🟢 Warm-up · *Size the system*

Given new links per day, the read:write ratio, years kept and bytes per record, return `(write QPS, read QPS, storage in TB)`, rounded to 1.

**Examples**

- `per_day = 100000000, ratio = 100, years = 5, record_bytes = 500` → `(1157.4, 115740.7, 91.2)`

In [ ]:
def shortener_math(per_day, ratio, years, record_bytes):
    """
    Given new links per day, the read:write ratio, years kept and bytes per
    record, return (write QPS, read QPS, storage in TB), rounded to 1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shortener_math, [
    ((100000000, 100, 5, 500), (1157.4, 115740.7, 91.2)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step-by-step arithmetic.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, compact.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Step-by-step arithmetic.

```python
def shortener_math(per_day, ratio, years, record_bytes):
    seconds = 24 * 60 * 60
    writes = per_day / seconds
    reads = writes * ratio
    records = per_day * 365 * years
    tb = records * record_bytes / 10 ** 12
    return (round(writes, 1), round(reads, 1), round(tb, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, compact.

```python
def shortener_math(per_day, ratio, years, record_bytes):
    w = per_day / 86400
    return (round(w, 1), round(w * ratio, 1), round(per_day * 365 * years * record_bytes / 1e12, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step-by-step arithmetic. That costs O(1) time and O(1) space."

"I can do better: Same, compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/napkin-maths-for-a-shortener)

<a id="q10"></a>
### Q10. Machine IDs Across Data Centres

🟢 Warm-up · *Pack and check the worker bits*

Snowflake's 10 machine bits are often 5 bits of data centre + 5 bits of worker. Return the machine id `dc << 5 | worker` for each `(dc, worker)` (or 'out of range' if either is ≥ 32), and the machine ids that appear more than once.

**Examples**

- `pairs = [(0, 1), (1, 1), (3, 31), (1, 1), (32, 0), (2, 40)]` → `([1, 33, 127, 33, 'out of range', 'out of range'], [33])`

In [ ]:
def machine_ids(pairs):
    """
    Snowflake's 10 machine bits are often 5 bits of data centre + 5 bits of
    worker. Return the machine id dc << 5 | worker for each (dc, worker) (or
    'out of range' if either is ≥ 32), and the machine ids that appear more than
    once.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(machine_ids, [
    (([(0, 1), (1, 1), (3, 31), (1, 1), (32, 0), (2, 40)],), ([1, 33, 127, 33, 'out of range', 'out of range'], [33])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute, then count duplicates with a list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension and a Counter.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compute, then count duplicates with a list.

```python
def machine_ids(pairs):
    ids = []
    for dc, w in pairs:
        if dc >= 32 or w >= 32:
            ids.append("out of range")
        else:
            ids.append(dc * 32 + w)
    dupes = []
    for i in ids:
        if i != "out of range" and ids.count(i) > 1 and i not in dupes:
            dupes.append(i)
    return (ids, sorted(dupes))
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension and a Counter.

```python
def machine_ids(pairs):
    ids = ["out of range" if dc >= 32 or w >= 32 else dc << 5 | w for dc, w in pairs]
    c = Counter(i for i in ids if i != "out of range")
    return (ids, sorted(i for i, n in c.items() if n > 1))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute, then count duplicates with a list. That costs O(n²) time and O(n) space."

"I can do better: A comprehension and a Counter. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/machine-ids-across-data-centres)

<a id="q11"></a>
### Q11. Sequential IDs Leak Your Numbers

🟢 Warm-up · *Why public IDs should be random*

A competitor creates an order on several days and notes its id. With sequential ids, the difference between consecutive observations reveals orders per day. Observations are `(day, id)`; return the average orders per day between each pair (rounded to 1).

**Examples**

- `observations = [(1, 10500), (8, 24500), (15, 45500), (16, 49000)]` → `[2000.0, 3000.0, 3500.0]`

In [ ]:
def id_leak(observations):
    """
    A competitor creates an order on several days and notes its id. With
    sequential ids, the difference between consecutive observations reveals
    orders per day. Observations are (day, id); return the average orders per
    day between each pair (rounded to 1).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(id_leak, [
    (([(1, 10500), (8, 24500), (15, 45500), (16, 49000)],), [2000.0, 3000.0, 3500.0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over neighbours.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`zip` over neighbours.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop over neighbours.

```python
def id_leak(observations):
    out = []
    for i in range(1, len(observations)):
        d0, id0 = observations[i - 1]
        d1, id1 = observations[i]
        out.append(round((id1 - id0) / (d1 - d0), 1))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `zip` over neighbours.

```python
def id_leak(observations):
    return [round((b - a) / (e - d), 1) for (d, a), (e, b) in zip(observations, observations[1:])]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over neighbours. That costs O(n) time and O(n) space."

"I can do better: `zip` over neighbours. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/sequential-ids-leak-your-numbers)

<a id="q12"></a>
### Q12. Hash, Then Handle Collisions

🟡 Interview standard · *Content-derived codes*

Take the first `k` hex characters of MD5(url). If that code is taken by a different URL, append '#1', '#2', … to the URL and rehash until free. The same URL always gets its existing code. Return the codes.

**Examples**

- `urls = ['a.com/1', 'a.com/2', 'a.com/3', 'a.com/1', 'b.com/9', 'b.com/8', 'c.com/x'], k = 1` → `['f', '8', '6', 'f', 'e', '7', '5']`

In [ ]:
import hashlib

def hash_codes(urls, k):
    """
    Take the first k hex characters of MD5(url). If that code is taken by a
    different URL, append '#1', '#2', … to the URL and rehash until free. The
    same URL always gets its existing code. Return the codes.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(hash_codes, [
    ((['a.com/1', 'a.com/2', 'a.com/3', 'a.com/1', 'b.com/9', 'b.com/8', 'c.com/x'], 1), ['f', '8', '6', 'f', 'e', '7', '5']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Search a list of (code, url) pairs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Dicts in both directions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Search a list of (code, url) pairs.

```python
def hash_codes(urls, k):
    taken, out = [], []
    for u in urls:
        existing = [c for c, url in taken if url == u]
        if existing:
            out.append(existing[0])
            continue
        attempt, salt = u, 0
        while True:
            code = hashlib.md5(attempt.encode()).hexdigest()[:k]
            if code not in [c for c, _ in taken]:
                break
            salt += 1
            attempt = f"{u}#{salt}"
        taken.append((code, u))
        out.append(code)
    return out
```

**Cost:** O(n² · r) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Dicts in both directions.

```python
def hash_codes(urls, k):
    by_code, by_url, out = {}, {}, []
    for u in urls:
        if u not in by_url:
            salt, code = 0, hashlib.md5(u.encode()).hexdigest()[:k]
            while code in by_code:
                salt += 1
                code = hashlib.md5(f"{u}#{salt}".encode()).hexdigest()[:k]
            by_code[code], by_url[u] = u, code
        out.append(by_url[u])
    return out
```

**Cost:** O(n · r) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Search a list of (code, url) pairs. That costs O(n² · r) time and O(n) space."

"I can do better: Dicts in both directions. That's O(n · r) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/hash-then-handle-collisions)

<a id="q13"></a>
### Q13. Hand Out ID Ranges

🟡 Interview standard · *Ticket server / range allocation*

A central counter hands each app server a block of `block` ids at a time; servers use their block locally without coordination. For a sequence of servers asking for one id each, return the ids issued.

**Examples**

- `requests = ['s1', 's1', 's2', 's1', 's2', 's3', 's1', 's1'], block = 3` → `[1, 2, 4, 3, 5, 7, 10, 11]`

In [ ]:
def id_ranges(requests, block):
    """
    A central counter hands each app server a block of block ids at a time;
    servers use their block locally without coordination. For a sequence of
    servers asking for one id each, return the ids issued.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(id_ranges, [
    ((['s1', 's1', 's2', 's1', 's2', 's3', 's1', 's1'], 3), [1, 2, 4, 3, 5, 7, 10, 11]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Lists of remaining ids per server.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep (next, end) per server.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Lists of remaining ids per server.

```python
def id_ranges(requests, block):
    counter, pools, out = 1, {}, []
    for s in requests:
        if not pools.get(s):
            pools[s] = list(range(counter, counter + block))
            counter += block
        out.append(pools[s].pop(0))
    return out
```

**Cost:** O(n · b) time, O(s · b) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Keep (next, end) per server.

```python
def id_ranges(requests, block):
    counter, state, out = 1, {}, []
    for s in requests:
        nxt, end = state.get(s, (0, 0))
        if nxt == end:
            nxt, end, counter = counter, counter + block, counter + block
        out.append(nxt)
        state[s] = (nxt + 1, end)
    return out
```

**Cost:** O(n) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Lists of remaining ids per server. That costs O(n · b) time and O(s · b) space."

"I can do better: Keep (next, end) per server. That's O(n) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/hand-out-id-ranges)

<a id="q14"></a>
### Q14. A Key-Generation Service

🟡 Interview standard · *Pre-made keys and what a crash wastes*

A KGS holds `pool` unused keys and gives servers batches of `batch`. Events are `('use', server)` (take one key; fetch a batch if the server has none) and `('crash', server)` (its unused keys are lost). Return `(keys used, keys wasted, keys left in the pool)`.

**Examples**

- `pool = 1000, batch = 10, events = [('use', 'a'), ('use', 'a'), ('use', 'b'), ('crash', 'a'), ('use', 'a'), ('use', 'b'), ('crash', 'b')]` → `(5, 16, 970)`

In [ ]:
def kgs(pool, batch, events):
    """
    A KGS holds pool unused keys and gives servers batches of batch. Events are
    ('use', server) (take one key; fetch a batch if the server has none) and
    ('crash', server) (its unused keys are lost). Return (keys used, keys
    wasted, keys left in the pool).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(kgs, [
    ((1000, 10, [('use', 'a'), ('use', 'a'), ('use', 'b'), ('crash', 'a'), ('use', 'a'), ('use', 'b'), ('crash', 'b')]), (5, 16, 970)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Lists of keys per server.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Counters per server.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Lists of keys per server.

```python
def kgs(pool, batch, events):
    held, used, wasted = {}, 0, 0
    for kind, s in events:
        if kind == "use":
            if not held.get(s):
                held[s] = list(range(batch))
                pool -= batch
            held[s].pop()
            used += 1
        else:
            wasted += len(held.get(s, []))
            held[s] = []
    return (used, wasted, pool)
```

**Cost:** O(e · b) time, O(s · b) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Counters per server.

```python
def kgs(pool, batch, events):
    held, used, wasted = defaultdict(int), 0, 0
    for kind, s in events:
        if kind == "crash":
            wasted, held[s] = wasted + held[s], 0
            continue
        if held[s] == 0:
            held[s], pool = batch, pool - batch
        held[s], used = held[s] - 1, used + 1
    return (used, wasted, pool)
```

**Cost:** O(e) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Lists of keys per server. That costs O(e · b) time and O(s · b) space."

"I can do better: Counters per server. That's O(e) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/a-key-generation-service)

<a id="q15"></a>
### Q15. Build a Snowflake ID

🟡 Interview standard · *Time + machine + sequence*

Compose 64-bit ids: `(ms − EPOCH) << 22 | machine << 12 | sequence` for each `(unix_ms, machine, sequence)`. IDs sort by time and need no coordination.

**Examples**

- `parts = [(1700000000000, 1, 0), (1700000000001, 1, 0), (1700000000001, 1, 1), (1700000000001, 2, 0), (1767225600000, 1023, 4095)]` → `[4096, 4198400, 4198401, 4202496, 281964602986594303]`

In [ ]:
def snowflake(parts):
    """
    Compose 64-bit ids: (ms − EPOCH) << 22 | machine << 12 | sequence for each
    (unix_ms, machine, sequence). IDs sort by time and need no coordination.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(snowflake, [
    (([(1700000000000, 1, 0), (1700000000001, 1, 0), (1700000000001, 1, 1), (1700000000001, 2, 0), (1767225600000, 1023, 4095)],), [4096, 4198400, 4198401, 4202496, 281964602986594303]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply by powers of two.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Shifts and ORs.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Multiply by powers of two.

```python
def snowflake(parts):
    out = []
    for ms, machine, seq in parts:
        out.append((ms - EPOCH) * 2 ** 22 + machine * 2 ** 12 + seq)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Shifts and ORs.

```python
def snowflake(parts):
    return [(ms - EPOCH) << 22 | m << 12 | s for ms, m, s in parts]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply by powers of two. That costs O(n) time and O(n) space."

"I can do better: Shifts and ORs. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/build-a-snowflake-id)

<a id="q16"></a>
### Q16. The Clock Went Backwards

🟡 Interview standard · *Never issue a duplicate*

An ID generator reads the clock for each request. If the clock is behind the last timestamp used (NTP adjusted it), keep using the last timestamp and continue its sequence instead of going back. Return `(timestamp, sequence)` per request.

**Examples**

- `readings = [100, 100, 101, 99, 99, 101, 102]` → `[(100, 0), (100, 1), (101, 0), (101, 1), (101, 2), (101, 3), (102, 0)]`

In [ ]:
def clock_back(readings):
    """
    An ID generator reads the clock for each request. If the clock is behind the
    last timestamp used (NTP adjusted it), keep using the last timestamp and
    continue its sequence instead of going back. Return (timestamp, sequence)
    per request.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(clock_back, [
    (([100, 100, 101, 99, 99, 101, 102],), [(100, 0), (100, 1), (101, 0), (101, 1), (101, 2), (101, 3), (102, 0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track last timestamp and sequence with if/elif.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max` of clock and last.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Track last timestamp and sequence with if/elif.

```python
def clock_back(readings):
    last, seq, out = -1, 0, []
    for t in readings:
        if t > last:
            last, seq = t, 0
        else:
            seq += 1
        out.append((last, seq))
    return out
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `max` of clock and last.

```python
def clock_back(readings):
    last, seq, out = -1, 0, []
    for t in readings:
        ts = max(t, last)
        seq = seq + 1 if ts == last else 0
        last = ts
        out.append((ts, seq))
    return out
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track last timestamp and sequence with if/elif. That costs O(n) time and O(1) space."

"I can do better: `max` of clock and last. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/the-clock-went-backwards)

<a id="q17"></a>
### Q17. Time-Sortable String IDs

🟡 Interview standard · *ULID-style timestamps*

Encode each millisecond timestamp as 10 characters of Crockford base32 (as ULIDs do). Return the codes and whether sorting the strings gives the same order as sorting the numbers — the reason ULIDs and UUIDv7 make good database keys.

**Examples**

- `times = [1700000000000, 1700000000001, 1600000000000, 1800000000000]` → `(['01HF7YAT00', '01HF7YAT01', '01EJ3PX000', '01MCC5RM00'], True)`

In [ ]:
CROCKFORD = "0123456789ABCDEFGHJKMNPQRSTVWXYZ"

def ulid_time(times):
    """
    Encode each millisecond timestamp as 10 characters of Crockford base32 (as
    ULIDs do). Return the codes and whether sorting the strings gives the same
    order as sorting the numbers — the reason ULIDs and UUIDv7 make good
    database keys.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ulid_time, [
    (([1700000000000, 1700000000001, 1600000000000, 1800000000000],), (['01HF7YAT00', '01HF7YAT01', '01EJ3PX000', '01MCC5RM00'], True)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Divide by 32 ten times.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Shift by 5 bits ten times.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Divide by 32 ten times.

```python
def ulid_time(times):
    codes = []
    for t in times:
        s = ""
        for _ in range(10):
            s = CROCKFORD[t % 32] + s
            t //= 32
        codes.append(s)
    same = [c for _, c in sorted(zip(times, codes))] == sorted(codes)
    return (codes, same)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Shift by 5 bits ten times.

```python
def ulid_time(times):
    codes = ["".join(CROCKFORD[(t >> (5 * i)) & 31] for i in range(9, -1, -1)) for t in times]
    return (codes, sorted(codes) == [c for _, c in sorted(zip(times, codes))])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Divide by 32 ten times. That costs O(n) time and O(n) space."

"I can do better: Shift by 5 bits ten times. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/time-sortable-string-ids)

<a id="q18"></a>
### Q18. When Do Random IDs Collide?

🟡 Interview standard · *The birthday paradox*

With ids of `bits` random bits, the chance of any collision among n ids ≈ 1 − e^(−n² / 2^(bits+1)). Return it (rounded to 4) for each n.

**Examples**

- `ns = [1000, 100000, 1000000, 100000000], bits = 40` → `[0.0, 0.0045, 0.3654, 1.0]`
- `ns = [1000000000], bits = 122` → `[0.0]`

In [ ]:
def birthday(ns, bits):
    """
    With ids of bits random bits, the chance of any collision among n ids ≈ 1 −
    e^(−n² / 2^(bits+1)). Return it (rounded to 4) for each n.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(birthday, [
    (([1000, 100000, 1000000, 100000000], 40), [0.0, 0.0045, 0.3654, 1.0]),
    (([1000000000], 122), [0.0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.expm1` for accuracy with tiny values.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop.

```python
def birthday(ns, bits):
    out = []
    for n in ns:
        out.append(round(1 - math.exp(-n * n / 2 ** (bits + 1)), 4))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `math.expm1` for accuracy with tiny values.

```python
def birthday(ns, bits):
    return [round(-math.expm1(-n * n / 2 ** (bits + 1)), 4) for n in ns]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(n) time and O(n) space."

"I can do better: `math.expm1` for accuracy with tiny values. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/when-do-random-ids-collide)

<a id="q19"></a>
### Q19. A Bloom Filter for Taken Codes

🟡 Interview standard · *Maybe yes, definitely no*

Insert `taken` codes into a Bloom filter of `m` bits using k hash functions `crc32(f'{i}:{code}') % m`. For each query return True ('maybe taken') or False ('definitely free'). False positives are possible; false negatives aren't.

**Examples**

- `taken = ['abc', 'xyz', 'q1w2', 'zz9'], m = 32, k = 3, queries = ['abc', 'zz9', 'new1', 'hello', 'q1w2', 'free7']` → `[True, True, False, False, True, False]`

In [ ]:
import zlib

def bloom(taken, m, k, queries):
    """
    Insert taken codes into a Bloom filter of m bits using k hash functions
    crc32(f'{i}:{code}') % m. For each query return True ('maybe taken') or
    False ('definitely free'). False positives are possible; false negatives
    aren't.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(bloom, [
    ((['abc', 'xyz', 'q1w2', 'zz9'], 32, 3, ['abc', 'zz9', 'new1', 'hello', 'q1w2', 'free7']), [True, True, False, False, True, False]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list of booleans.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

An integer used as a bit set.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A list of booleans.

```python
def bloom(taken, m, k, queries):
    bits = [False] * m
    for c in taken:
        for i in range(k):
            bits[zlib.crc32(f"{i}:{c}".encode()) % m] = True
    out = []
    for q in queries:
        maybe = True
        for i in range(k):
            if not bits[zlib.crc32(f"{i}:{q}".encode()) % m]:
                maybe = False
        out.append(maybe)
    return out
```

**Cost:** O((t + q) · k) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** An integer used as a bit set.

```python
def bloom(taken, m, k, queries):
    def positions(s):
        return [zlib.crc32(f"{i}:{s}".encode()) % m for i in range(k)]
    mask = 0
    for c in taken:
        for p in positions(c):
            mask |= 1 << p
    return [all(mask >> p & 1 for p in positions(q)) for q in queries]
```

**Cost:** O((t + q) · k) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list of booleans. That costs O((t + q) · k) time and O(m) space."

"I can do better: An integer used as a bit set. That's O((t + q) · k) time and O(m) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `taken = ['a'], m = 16, k = 2, queries = ['a', 'b']`

1. positions() returns [5, 2]
2. positions() returns [5, 2]
3. positions() returns [15, 8]
4. GeneratorExit raised: 
5. bloom() returns [True, False]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/a-bloom-filter-for-taken-codes)

<a id="q20"></a>
### Q20. Canonicalise URLs Before Shortening

🟡 Interview standard · *One long URL, one code*

Lower-case the scheme and host, drop default ports (:80 for http, :443 for https), drop the #fragment, use '/' for an empty path, and sort query parameters. Return the canonical URLs and how many distinct ones there are.

**Examples**

- `urls = ['HTTPS://Example.com:443/a?b=2&a=1#top', 'https://example.com/a?a=1&b=2', 'http://example.com:80', 'http://EXAMPLE.com/', 'https://example.com:8443/a']` → `(['https://example.com/a?a=1&b=2', 'https://example.com/a?a=1&b=2', 'http://example.com/', 'http://example.com/', 'https://example.com:8443/a'], 3)`

In [ ]:
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

def canonical(urls):
    """
    Lower-case the scheme and host, drop default ports (:80 for http, :443 for
    https), drop the #fragment, use '/' for an empty path, and sort query
    parameters. Return the canonical URLs and how many distinct ones there are.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(canonical, [
    ((['HTTPS://Example.com:443/a?b=2&a=1#top', 'https://example.com/a?a=1&b=2', 'http://example.com:80', 'http://EXAMPLE.com/', 'https://example.com:8443/a'],), (['https://example.com/a?a=1&b=2', 'https://example.com/a?a=1&b=2', 'http://example.com/', 'http://example.com/', 'https://example.com:8443/a'], 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

String splitting by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`urlsplit` / `urlunsplit`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** String splitting by hand.

```python
def canonical(urls):
    out = []
    for u in urls:
        u = u.split("#")[0]
        scheme, rest = u.split("://", 1)
        scheme = scheme.lower()
        if "?" in rest:
            rest, query = rest.split("?", 1)
        else:
            query = ""
        if "/" in rest:
            host, path = rest.split("/", 1)
            path = "/" + path
        else:
            host, path = rest, "/"
        host = host.lower()
        if (scheme == "http" and host.endswith(":80")) or (scheme == "https" and host.endswith(":443")):
            host = host.rsplit(":", 1)[0]
        params = sorted(query.split("&")) if query else []
        out.append(f"{scheme}://{host}{path}" + ("?" + "&".join(params) if params else ""))
    return (out, len(set(out)))
```

**Cost:** O(n · L) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `urlsplit` / `urlunsplit`.

```python
def canonical(urls):
    def canon(u):
        p = urlsplit(u)
        scheme, host = p.scheme.lower(), p.hostname
        port = f":{p.port}" if p.port and p.port != {"http": 80, "https": 443}[scheme] else ""
        query = urlencode(sorted(parse_qsl(p.query)))
        return urlunsplit((scheme, host + port, p.path or "/", query, ""))
    out = [canon(u) for u in urls]
    return (out, len(set(out)))
```

**Cost:** O(n · L) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: String splitting by hand. That costs O(n · L) time and O(n) space."

"I can do better: `urlsplit` / `urlunsplit`. That's O(n · L) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/ids/canonicalise-urls-before-shortening)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. 301 or 302 for redirects?</b></summary>

301 is permanent and cached by browsers, so repeat clicks never reach you — less load but no analytics. 302 (or 307) is temporary, so every click is counted.

</details>

<details>
<summary><b>2. How do you avoid two servers generating the same short code?</b></summary>

Use a central counter with ranges per server, a key-generation service with pre-allocated keys, or hash with a uniqueness check (a conditional insert that fails if the code exists).

</details>

<details>
<summary><b>3. How would you scale reads for a shortener?</b></summary>

Cache hot codes in Redis and at the CDN edge, use read replicas or a key-value store sharded by code, and keep the redirect path free of synchronous analytics writes.

</details>

<details>
<summary><b>4. Why not use UUIDs as primary keys everywhere?</b></summary>

Random UUIDs scatter inserts across the B-tree, causing page splits and poor cache use. Time-ordered IDs (UUIDv7, ULID, Snowflake) insert near the end of the index.

</details>

<details>
<summary><b>5. What if Snowflake's clock goes backwards?</b></summary>

Refuse to issue ids or wait until the clock passes the last timestamp, or keep using the last timestamp with the sequence. Never issue ids with an earlier timestamp.

</details>

<details>
<summary><b>6. Why shouldn't public ids be sequential?</b></summary>

They reveal volumes (orders per day) and make enumeration attacks easy. Expose random or encoded ids and keep sequential keys internal.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Design a URL shortener
- [ ] I can explain: Generating short codes
- [ ] I can explain: Unique IDs at scale

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) The Clock Went Backwards
- [ ] [Q17](#q17) Time-Sortable String IDs
- [ ] [Q18](#q18) When Do Random IDs Collide?
- [ ] [Q19](#q19) A Bloom Filter for Taken Codes
- [ ] [Q20](#q20) Canonicalise URLs Before Shortening

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.